# E4 · Как модель исполняет полудуплексный протокол

Проверяем, насколько точно языковая модель выполняет **заранее объяснённый алгоритм обмена**.
Алиса получает битовую строку `x` длины `n`, Боб — строку `y` той же длины.
Каждый видит только свой вход и локальные наблюдения. Оба должны вычислить один и тот же бит `f(x, y)`.
Программный эталон задаёт правильные действия и служит для проверки модели.

| `task` | Функция | Ответ 1 |
|---|---|---|
| `ip` | Inner Product — скалярное произведение по модулю 2 | Нечётное число позиций с двумя единицами |
| `eq` | Equality — равенство | `x == y` |
| `disj` | Disjointness — непересечение | Нет общих единичных позиций |

В раунде оба независимо выбирают `SEND_0`, `SEND_1` или `LISTEN`.
Один отправитель — слушатель получает бит; два отправителя — биты теряются,
а столкновение им не сообщается. Когда оба слушают, `channel="silence"` возвращает
`SILENCE`, `channel="zero"` — обычный 0. Отправитель не получает наблюдения.
В конце оба одновременно выдают `ANSWER_0` или `ANSWER_1`: это локальное решение без передачи и без дополнительного раунда.

**Два режима E4:**

- `episode`: проводим полную игру. У каждой роли отдельная беседа; следующий ход
  опирается на реальные наблюдения. Измеряем верность обоих ответов, точное исполнение и число раундов.
- `step`: запрашиваем следующий токен для каждого уникального локального состояния
  эталонных траекторий. Каждый запрос независим; измеряем точность действий и ответов.

Правильный итоговый бит и точное следование алгоритму — отдельные метрики.
Описание проекта, источники и запуск из терминала собраны в [README.md](README.md).

### Как генерируются входы

Всего есть `4**n` пар строк. `seed` фиксирует генератор `random.Random`,
`case_limit` задаёт число разных пар, `repeats` — число повторов на тех же данных.

- `uniform`: равномерно выбираем пары без повторений. Индекс пары раскладывается
  на `x, y = divmod(i, 2**n)`, оба числа записываются ровно `n` битами.
- `eq_balanced`: для EQ выбираем поровну равных и неравных пар, затем перемешиваем.
  При нечётном размере равных на одну больше: для 15 пар — 8 равных и 7 неравных.
  Это отдельное распределение: при `uniform` вероятность `x == y` равна `2**(-n)`.
- Для одной заданной пары: `case_limit=None, case=("00111", "00100")` при `n=5`.

При одинаковых настройках выборки смена моделей, алгоритма, формата входа или
объяснения сохраняет пары и их порядок. `input_format` меняет только представление
строки в промпте: `bits`, `bit_list` или `positions` (позиции единиц, с 1 слева).
В разделе 2 выведем выбранные пары до обращения к модели.

### Какие алгоритмы используем

`task` выбирает функцию, `channel` — канал, `protocol` — алгоритм обмена.
`baseline` использует универсальный обмен, `specialized` выбирает одну из трёх конструкций по функции и каналу.

| Реализация | Выбор | Идея | Максимум раундов |
|---|---|---|---|
| `send_x_then_result` | `baseline`; любая функция и канал | Алиса передаёт весь `x`, Боб вычисляет функцию и возвращает бит | `n+1` |
| `ip_silence` | `specialized`, IP / `silence` | Обрабатываем пары битов, накапливаем локальные биты, объединяем их через XOR | `ceil(n/2)+2` |
| `eq_silence` | `specialized`, EQ / `silence` | Кодируем входы в основании 5; проверяем символы и число успешных приёмов | `k+w+2` |
| `eq_zero` | `specialized`, EQ / `zero` | Кодируем входы в основании 3; обмениваемся счётчиками отправок и проверяем несовпадения | `k+2*w+1` |

Для EQ/silence: `k=ceil(n/log_2(5))`, `w=ceil(log_3(k+1))`.
Для EQ/zero: `k=ceil(n/log_2(3))`, `w=ceil(log_2(k+1))`.
В стоимость включены завершающий обмен и счётчики. EQ/silence может закончиться раньше,
за `k+1` раундов, если обнаружено несовпадение. При малых `n` специализированная схема может быть дороже базовой.
Для DISJ и для IP/zero доступен только `baseline`.
Источники: IP — Dementiev et al., теорема 3; EQ — Hoover et al., теоремы 6 и 10
(ссылки в [README](README.md#алгоритмы)).

### Какой промпт получает модель и что запускаем

Шаблон `render_prompt` из `halfduplex/prompts.py` содержит роль, функцию,
правила канала, лимит раундов, описание выбранного алгоритма, свой вход и локальную историю.
Модель должна вернуть ровно один токен: `SEND_0`, `SEND_1`, `LISTEN`, `ANSWER_0` или `ANSWER_1`.
Чужой вход и правильный ответ в инструкцию не передаются.

`explanation="prose"` даёт словесное объяснение; `"pseudocode"` дополнительно включает
локальное правило на Python. В `episode` начальный промпт открывает беседу роли,
а следующие сообщения передают её последнее действие и наблюдение. В `step` полная
заданная история включается в каждый независимый запрос. В разделе 2 покажем **фактический стартовый промпт Алисы**.

По умолчанию запускаем IP с тишиной, алгоритм `specialized`, `n=5`, 15 пар,
`seed=42`, один повтор, режим `episode`; обе роли исполняет Codex.
При `allow_python=True` доступен только изолированный Python без сети и файлов проекта.
Терминал, веб-поиск, приложения и дополнительные агенты отключены.

**Порядок работы:** настройки → входы и промпт → проверка программного эталона →
запуск → отчёт и графики. Для запуска включи `RUN_EXPERIMENT=True` в разделе 5.
С флагами по умолчанию `Run All` выполняет подготовку без обращений к моделям.
Для LLM нужны Codex CLI с файловым входом `codex login`, Linux/WSL и bubblewrap;
для графиков — зависимости из `requirements-plots.txt` в окружении ядра.

## 1. Настройки

Выбери функцию, канал и алгоритм отдельно. Смена `protocol` сохраняет `task` и `channel`.
Для EQ можно задать `sampling="eq_balanced"`; при маленьком `n` уменьши `case_limit`
до допустимого числа разных пар (`4**n`, а для сбалансированной выборки должно хватать равных пар).
`model=None` использует встроенный выбор CLI; для сравнения моделей задай точные имена у каждой роли.
Чтобы выполнить обмен программно, выбери у обеих ролей `backend="python"`.

In [ ]:
from dataclasses import replace
from pathlib import Path
import json
from halfduplex import (
    RunConfig, PlayerConfig, run_experiment, verify_reference,
    analyze_file, plot_results, audit_tools, input_pairs,
)
from halfduplex.analysis import reference_player
from halfduplex.core import View, play
from halfduplex.prompts import render_prompt


In [ ]:
config = RunConfig(
    task="ip",                 # ip, eq, disj — вычисляемая функция
    channel="silence",         # silence, zero — канал
    protocol="specialized",    # baseline, specialized — алгоритм обмена
    n=5,
    experiment="episode",  # episode: полный обмен; step: эталонные локальные истории
    alice=PlayerConfig(backend="codex", model=None, allow_python=True),
    bob=PlayerConfig(backend="codex", model=None, allow_python=True),
    explanation="prose",  # prose или pseudocode
    input_format="bits",  # bits, bit_list, positions
    sampling="uniform",   # для EQ можно eq_balanced
    case_limit=15, seed=42, repeats=1,
    output_dir="runs",
)
print("Функция:", config.task, "Канал:", config.channel, "Алгоритм:", config.protocol)
print("Реализация:", config.implementation.name, "Лимит раундов:", config.max_rounds)


## 2. Входы и пример промпта

Сначала показываем первые пять пар и состав выборки. Это общий обзор для исследователя;
игроку будет передан только его собственный вход. Пары воспроизводятся по `seed`.

In [ ]:
pairs = input_pairs(config)
print("Пар:", len(pairs), "Распределение:", config.sampling, "Seed:", config.seed)
print("Первые пары (x, y):")
print(json.dumps(pairs[:5], ensure_ascii=False, indent=2))
if config.task == "eq":
    equal_count = sum(x == y for x, y in pairs)
    print("Равных:", equal_count, "Неравных:", len(pairs) - equal_count)


### Стартовый промпт Алисы

Ниже тот же шаблон, который использует адаптер модели, с текущими настройками
и первым входом Алисы. Для Боба подставляются его роль и его собственный вход.
Вывод промпта не вызывает модель.

In [ ]:
prompt_view = View("Alice", pairs[0][0])
example_prompt = render_prompt(config, prompt_view, persistent=config.experiment == "episode")
print(example_prompt)


## 3. Проверка программного эталона

Проверяем оба ответа и согласованную остановку на всех `4**n` парах, измеряем
максимум раундов `c_max`. Это проверка выбранного алгоритма без LLM, отдельно от
выборки из `case_limit` пар. При слишком большом `n` проверка остановится по лимиту перебора.

In [ ]:
reference_check = verify_reference(config)
print(json.dumps(reference_check, ensure_ascii=False, indent=2))


### Один эталонный эпизод

На первой выбранной паре показываем весь обмен, наблюдения и ответы обоих игроков.
Этот общий журнал нужен исследователю; модели получают только свои локальные наблюдения.

In [ ]:
example_x, example_y = pairs[0]
reference = reference_player(config)
example = play(config.n, example_x, example_y, reference, reference,
               channel=config.channel, max_rounds=config.implementation.rounds(config.n))
print("x =", example_x, "y =", example_y)
print(json.dumps(example.record(), ensure_ascii=False, indent=2))


## 4. Проверка инструмента Python (необязательно)

`RUN_TOOL_AUDIT=True` делает один реальный запрос к модели: она должна вызвать Python,
вычислить 42 и подтвердить отсутствие сети и доступа к файлам хоста.
Аудит сохраняется в отдельный каталог `runs/`. Для обычного прогона этот флаг можно
оставить `False`; доступность изоляции проверяется автоматически при запуске эксперимента.

In [ ]:
RUN_TOOL_AUDIT = False
tool_audit = audit_tools(config.alice, output_dir=config.output_dir) if RUN_TOOL_AUDIT else None
if tool_audit:
    print(json.dumps(tool_audit, ensure_ascii=False, indent=2))


## 5. Запуск эксперимента

Поставь `RUN_EXPERIMENT=True` и выполни ячейку. Запустится конфигурация из раздела 1:
при `backend="codex"` будут реальные запросы к моделям, при `backend="python"` — программный эталон.
При `False` запросы не выполняются.

В `episode` каждая пара и каждый повтор получают новые отдельные беседы Алисы и Боба.
В `step` каждое уникальное локальное состояние проверяется независимым запросом;
поэтому число запросов отличается от числа пар. Данные сохраняются в `runs/e4_.../`.
В конце выводятся метрики: `joint_accuracy` и `exact_execution` для полной игры,
`next_action_accuracy`, `output_accuracy`, `decision_accuracy` для отдельных решений.

In [ ]:
RUN_EXPERIMENT = False
result = run_experiment(config) if RUN_EXPERIMENT else None
if result:
    print("Исходный журнал:", result.log_path)
    print("Отчёт:", result.report_path)


## 6. Отчёт по сохранённому журналу

Пересчёт читает записанные решения и сверяет их с программным эталоном без обращений к моделям.
`journal.jsonl` содержит исходные данные, `report.jsonl` — ошибки и метрики,
`summary.json` — сводку; в `calls/` находятся промпты и ответы.
Поддерживаются также журналы старого пилота. Можно указать свой `saved_log_path`.

In [ ]:
saved_log_path = result.log_path if result else None
# saved_log_path = Path("results/experiment4_episode_20261109T221631_854494Z.jsonl")
if saved_log_path is not None:
    saved_summary = analyze_file(saved_log_path)
    print(json.dumps(saved_summary, ensure_ascii=False, indent=2))


## 7. Графики

Поставь `PLOT_RESULTS=True`, если выбран `saved_log_path`. Слева будет точность,
справа — локальные ошибки по ролям. График сохранится рядом с журналом в `plots.png`.
Нужен matplotlib из `requirements-plots.txt`; построение не вызывает модель.

In [ ]:
PLOT_RESULTS = False
if PLOT_RESULTS and saved_log_path is not None:
    graph_path = Path(saved_log_path).with_name("plots.png")
    figure = plot_results(saved_log_path, save_path=graph_path)
    print("График:", graph_path)
